<a href="https://colab.research.google.com/github/22104071/Financial-Transaction-Data-Engineering-/blob/main/Fraud_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

np.random.seed(42)
random.seed(42)

In [25]:
df = pd.read_csv('/content/fraud_dataset.csv')

In [26]:
df.head()

,index,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0
1,1,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0
2,2,2019-01-01 00:07:27,Kiehn Inc,grocery_pos,96.29,Grenada,CA,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,41.657520,-122.230347,0
3,3,2019-01-01 00:09:03,Beier-Hyatt,shopping_pos,7.77,High Rolls Mountain Park,NM,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,32.863258,-106.520205,0
4,4,2019-01-01 00:21:32,Bruen-Yost,misc_pos,6.85,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",1967-08-02,f3c43d336e92a44fc2fb67058d5949e3,43.753735,-111.454923,0


In [27]:
df.shape

(6666, 16)

In [28]:
df.describe()

,index,amt,lat,long,city_pop,merch_lat,merch_long,is_fraud
count,6666.000000,6666.000000,6666.000000,6666.000000,6.666000e+03,6666.000000,6666.000000,6666.000000
mean,3332.500000,72.151169,39.719401,-110.812883,1.074933e+05,39.718273,-110.822444,0.009451
std,1924.452779,125.781507,5.319824,13.035408,2.887422e+05,5.349734,13.047740,0.096763
min,0.000000,1.010000,20.027100,-165.672300,4.600000e+01,19.040141,-166.629875,0.000000
25%,1666.250000,10.000000,36.670400,-120.093600,4.710000e+02,36.647607,-119.846388,0.000000
50%,3332.500000,47.040000,39.599400,-111.098500,1.645000e+03,39.538012,-111.013610,0.000000
75%,4998.750000,85.232500,41.696400,-101.136000,3.543900e+04,42.171323,-100.556067,0.000000
max,6665.000000,3178.510000,65.689900,-89.628700,2.383912e+06,66.659242,-88.927438,1.000000


In [29]:
df['is_fraud'].value_counts()
df['is_fraud'].value_counts(normalize=True)*100

,proportion
is_fraud,
0,99.054905
1,0.945095


In [30]:
unique_customers = df[['dob','city','state','job']].drop_duplicates()

unique_customers = unique_customers.reset_index(drop=True)

unique_customers['customer_id'] = [
    f"CUST{i+1:05d}"
    for i in range(len(unique_customers))
]

In [31]:
customer_df = unique_customers.copy()

In [32]:
customer_df['account_type'] = np.random.choice(
    ['Savings','Current','Premium'],
    size=len(customer_df)
)

customer_df['credit_score'] = np.random.randint(
    500,
    850,
    size=len(customer_df)
)

customer_df['account_age_years'] = np.random.randint(
    1,
    15,
    size=len(customer_df)
)

In [33]:
df = df.merge(
    customer_df[
        ['customer_id','dob','city','state','job']
    ],
    on=['dob','city','state','job'],
    how='left'
)

In [34]:
df['transaction_id'] = [
    f"TXN{i+1:07d}"
    for i in range(len(df))
]

In [38]:
df['trans_date_trans_time'] = pd.to_datetime(
    df['trans_date_trans_time']
)
df['dob'] = pd.to_datetime(
    df['dob']
)
df.dtypes

,0
index,int64
trans_date_trans_time,datetime64[ns]
merchant,object
category,object
amt,float64
city,object
state,object
lat,float64
long,float64
city_pop,int64


In [39]:
#Hour
df['hour'] = (
    df['trans_date_trans_time']
    .dt.hour
)

In [40]:
#Day
df['day_of_week'] = (
    df['trans_date_trans_time']
    .dt.day_name()
)

In [42]:
#Age
current_year = 2026

df['age'] = (
    current_year -
    pd.to_datetime(df['dob']).dt.year
)

In [43]:
customers = customer_df.copy()

In [44]:
customers.head()

,dob,city,state,job,customer_id,account_type,credit_score,account_age_years
0,1978-06-21,Orient,WA,Special educational needs teacher,CUST00001,Premium,633,3
1,1962-01-19,Malad City,ID,Nature conservation officer,CUST00002,Savings,783,12
2,1945-12-21,Grenada,CA,Systems analyst,CUST00003,Premium,527,9
3,1967-08-30,High Rolls Mountain Park,NM,Naval architect,CUST00004,Premium,607,5
4,1967-08-02,Freedom,WY,"Education officer, museum",CUST00005,Savings,543,8


In [45]:
transactions = df[
    [
        'transaction_id',
        'customer_id',
        'trans_date_trans_time',
        'merchant',
        'category',
        'amt',
        'lat',
        'long',
        'merch_lat',
        'merch_long',
        'is_fraud'
    ]
]

In [46]:
customers.to_csv(
    'customers_clean.csv',
    index=False
)

transactions.to_csv(
    'transactions_clean.csv',
    index=False
)

df.to_csv(
    'master_dataset.csv',
    index=False
)